In [ ]:
using BenchmarkTools
using InteractiveUtils
using LoopVectorization
using Profile
using ProfileSVG
using JSON

const raw = JSON.parse(read("particles.json", String))

const dt = 1e-3
const timesteps = 5

nothing

In [ ]:
raw

In [1]:
# Dane jako wektor słowników
module I1
export Particles, advance

struct Particles
    data::Vector{Any}
end

function advance(particles::Particles, dt)
    data = particles.data
    for row in data
        row["x"] = row["x"] + row["vx"] * dt
        row["y"] = row["y"] + row["vy"] * dt
        row["z"] = row["z"] + row["vz"] * dt
    end
    return particles
end

end

Main.I1

In [ ]:
p1 = I1.Particles(deepcopy(raw));

In [ ]:
@btime for _ in 1:timesteps
    I1.advance($p1, $dt)
end

In [ ]:
Profile.clear()
@profile for _ in 1:(100 * timesteps)
    I1.advance(p1, dt)
end
ProfileSVG.view()


In [ ]:
@code_warntype I1.advance(p1, dt)

In [ ]:
# Dane jako AoS
module I2
export Particle, Particles, advance

mutable struct Particle
    x; y; z
    vx; vy; vz
end

struct Particles
    data::Vector{Particle}
end

function advance(particles::Particles, dt)
    current = particles.data
    next = Vector{Particle}(undef, length(current))
    for i in eachindex(current)
        p = current[i]
        next[i] = Particle(
            p.x + p.vx * dt,
            p.y + p.vy * dt,
            p.z + p.vz * dt,
            p.vx,
            p.vy,
            p.vz,
        )
    end
    return Particles(next)
end

end


In [ ]:
p2 = I2.Particles([I2.Particle(row["x"], row["y"], row["z"], row["vx"], row["vy"], row["vz"]) for row in raw]);


In [ ]:
@btime for _ in 1:timesteps
    I2.advance($p2, $dt)
end

In [ ]:
Profile.clear()
@profile for _ in 1:(100 * timesteps)
    I2.advance(p2, dt)
end
ProfileSVG.view()


In [ ]:
@code_warntype I2.advance(p2, dt)


In [ ]:
# Dane jako otypowane AoS
module I3
export Particle, Particles, advance

mutable struct Particle
    x::Float64
    y::Float64
    z::Float64
    vx::Float64
    vy::Float64
    vz::Float64
end

struct Particles
    data::Vector{Particle}
end

function advance(particles::Particles, dt)
    current = particles.data
    next = Vector{Particle}(undef, length(current))
    for i in eachindex(current)
        p = current[i]
        next[i] = Particle(
            p.x + p.vx * dt,
            p.y + p.vy * dt,
            p.z + p.vz * dt,
            p.vx,
            p.vy,
            p.vz,
        )
    end
    return Particles(next)
end

end


In [ ]:
p3 = I3.Particles([
    I3.Particle(
        Float64(row["x"]), Float64(row["y"]), Float64(row["z"]),
        Float64(row["vx"]), Float64(row["vy"]), Float64(row["vz"]),
    )
    for row in raw
]);

In [ ]:
@btime for _ in 1:timesteps
    state = I3.advance($p3, $dt)
end

In [ ]:
Profile.clear()
@profile for _ in 1:(100 * timesteps)
    state = I3.advance(p3, dt)
end
ProfileSVG.view()


In [ ]:
@code_warntype I3.advance(p3, dt)


In [ ]:
# Struktura niemutowalna
module I4
export Particle, Particles, advance

struct Particle
    x::Float64
    y::Float64
    z::Float64
    vx::Float64
    vy::Float64
    vz::Float64
end

struct Particles
    data::Vector{Particle}
end

function advance(particles::Particles, dt)
    current = particles.data
    next = Vector{Particle}(undef, length(current))
    for i in eachindex(current)
        p = current[i]
        next[i] = Particle(
            p.x + p.vx * dt,
            p.y + p.vy * dt,
            p.z + p.vz * dt,
            p.vx,
            p.vy,
            p.vz,
        )
    end
    return Particles(next)
end

end


In [ ]:
p4 = I4.Particles([
    I4.Particle(
        Float64(row["x"]), Float64(row["y"]), Float64(row["z"]),
        Float64(row["vx"]), Float64(row["vy"]), Float64(row["vz"]),
    )
    for row in raw
]);

In [ ]:
@btime for _ in 1:timesteps
    state = I4.advance($p4, $dt)
end


In [ ]:
Profile.clear()
@profile for _ in 1:(100 * timesteps)
    state = I4.advance(p4, dt)
end
ProfileSVG.view()


In [ ]:
@code_warntype I4.advance(p4, dt)

In [ ]:
# Usunięcie alokacji
module I5
export Particle, Particles, advance

mutable struct Particle
    x::Float64; y::Float64; z::Float64
    vx::Float64; vy::Float64; vz::Float64
end

struct Particles
    data::Vector{Particle}
end

function advance(particles::Particles, dt)
    data = particles.data
    for i in eachindex(data)
        p = data[i]
        p.x += p.vx * dt
        p.y += p.vy * dt
        p.z += p.vz * dt
    end
    return particles
end

end


In [ ]:
p5 = I5.Particles([
    I5.Particle(
        Float64(row["x"]), Float64(row["y"]), Float64(row["z"]),
        Float64(row["vx"]), Float64(row["vy"]), Float64(row["vz"]),
    )
    for row in raw
]);


In [ ]:
@btime for _ in 1:timesteps
    I5.advance($p5, $dt)
end


In [ ]:
Profile.clear()
@profile for _ in 1:(100 * timesteps)
    I5.advance(p5, dt)
end
ProfileSVG.view()


In [ ]:
@code_warntype I5.advance(p5, dt)


In [ ]:
# Dane jako SoA
module I6
export Particles, advance_allocating, advance

struct Particles
    x::Vector{Float64}
    y::Vector{Float64}
    z::Vector{Float64}
    vx::Vector{Float64}
    vy::Vector{Float64}
    vz::Vector{Float64}
end

function advance_allocating(particles::Particles, dt)
    x, y, z = particles.x, particles.y, particles.z
    vx, vy, vz = particles.vx, particles.vy, particles.vz
    x = x .+ vx .* dt
    y = y .+ vy .* dt
    z = z .+ vz .* dt
    return Particles(x, y, z, vx, vy, vz)
end

function advance(particles::Particles, dt)
    x, y, z = particles.x, particles.y, particles.z
    vx, vy, vz = particles.vx, particles.vy, particles.vz
    x .= x .+ vx .* dt
    y .= y .+ vy .* dt
    z .= z .+ vz .* dt
    return particles
end

end


In [ ]:
p6 = I6.Particles(
    Float64[row["x"] for row in raw],
    Float64[row["y"] for row in raw],
    Float64[row["z"] for row in raw],
    Float64[row["vx"] for row in raw],
    Float64[row["vy"] for row in raw],
    Float64[row["vz"] for row in raw],
);


In [ ]:
@btime for _ in 1:timesteps
    state = I6.advance_allocating($p6, $dt)
end


In [ ]:

@btime for _ in 1:timesteps
    I6.advance($p6, $dt)
end


In [ ]:
Profile.clear()
@profile for _ in 1:(100 * timesteps)
    state = I6.advance_allocating(p6, dt)
end
ProfileSVG.view()


In [ ]:
Profile.clear()
@profile for _ in 1:(100 * timesteps)
    state = I6.advance(p6, dt)
end
ProfileSVG.view()

In [ ]:
@code_warntype I6.advance_allocating(p6, dt)


In [ ]:
# Dane jako SoA - jawna pętla 
module I7
export Particles, advance

struct Particles
    x::Vector{Float64}
    y::Vector{Float64}
    z::Vector{Float64}
    vx::Vector{Float64}
    vy::Vector{Float64}
    vz::Vector{Float64}
end

function advance(particles::Particles, dt)
    x, y, z = particles.x, particles.y, particles.z
    vx, vy, vz = particles.vx, particles.vy, particles.vz
    for i in eachindex(x)
        x[i] += vx[i] * dt
        y[i] += vy[i] * dt
        z[i] += vz[i] * dt
    end
    return particles
end

end


In [ ]:
p7 = I7.Particles(
    Float64[row["x"] for row in raw],
    Float64[row["y"] for row in raw],
    Float64[row["z"] for row in raw],
    Float64[row["vx"] for row in raw],
    Float64[row["vy"] for row in raw],
    Float64[row["vz"] for row in raw],
);


In [ ]:
@btime for _ in 1:timesteps
    I7.advance($p7, $dt)
end


In [ ]:
Profile.clear()
@profile for _ in 1:(1000 * timesteps)
    I7.advance(p7, dt)
end
ProfileSVG.view()


In [ ]:
@code_warntype I7.advance(p7, dt)


In [ ]:
# Pojedyncza precyzja
module I8
export Particles, advance

struct Particles
    x::Vector{Float32}
    y::Vector{Float32}
    z::Vector{Float32}
    vx::Vector{Float32}
    vy::Vector{Float32}
    vz::Vector{Float32}
end

function advance(particles::Particles, dt)
    Dt = Float32(dt)
    x, y, z = particles.x, particles.y, particles.z
    vx, vy, vz = particles.vx, particles.vy, particles.vz
    for i in eachindex(x)
        x[i] += vx[i] * Dt
        y[i] += vy[i] * Dt
        z[i] += vz[i] * Dt
    end
    return particles
end

end


In [ ]:
p8 = I8.Particles(
    Float32[row["x"] for row in raw],
    Float32[row["y"] for row in raw],
    Float32[row["z"] for row in raw],
    Float32[row["vx"] for row in raw],
    Float32[row["vy"] for row in raw],
    Float32[row["vz"] for row in raw],
);


In [ ]:
@btime for _ in 1:timesteps
    I8.advance($p8, $dt)
end


In [ ]:
Profile.clear()
@profile for _ in 1:(100 * timesteps)
    I8.advance(p8, dt)
end
ProfileSVG.view()


In [ ]:
@code_warntype I8.advance(p8, dt)


In [ ]:
# Makro @inbounds
module I9
export Particles, advance

struct Particles
    x::Vector{Float32}
    y::Vector{Float32}
    z::Vector{Float32}
    vx::Vector{Float32}
    vy::Vector{Float32}
    vz::Vector{Float32}
end

function advance(particles::Particles, dt)
    Dt = Float32(dt)
    x, y, z = particles.x, particles.y, particles.z
    vx, vy, vz = particles.vx, particles.vy, particles.vz
    @inbounds for i in eachindex(x)
        x[i] += vx[i] * Dt
        y[i] += vy[i] * Dt
        z[i] += vz[i] * Dt
    end
    return particles
end

end


In [ ]:
p9 = I9.Particles(
    Float32[row["x"] for row in raw],
    Float32[row["y"] for row in raw],
    Float32[row["z"] for row in raw],
    Float32[row["vx"] for row in raw],
    Float32[row["vy"] for row in raw],
    Float32[row["vz"] for row in raw],
);


In [ ]:
@btime for _ in 1:timesteps
    I9.advance($p9, $dt)
end


In [ ]:
Profile.clear()
@profile for _ in 1:(1000 * timesteps)
    I9.advance(p9, dt)
end
ProfileSVG.view()


In [ ]:
# @code_llvm I9.advance(p9, dt)

In [ ]:
# Makra @simd i @turbo 
module I10
using LoopVectorization
export Particles, advance, advance_turbo

struct Particles
    x::Vector{Float32}
    y::Vector{Float32}
    z::Vector{Float32}
    vx::Vector{Float32}
    vy::Vector{Float32}
    vz::Vector{Float32}
end

function advance(particles::Particles, dt)
    Dt = Float32(dt)
    x, y, z = particles.x, particles.y, particles.z
    vx, vy, vz = particles.vx, particles.vy, particles.vz
    @inbounds @simd for i in eachindex(x)
        x[i] += vx[i] * Dt
        y[i] += vy[i] * Dt
        z[i] += vz[i] * Dt
    end
    return particles
end

function advance_turbo(particles::Particles, dt)
    Dt = Float32(dt)
    x, y, z = particles.x, particles.y, particles.z
    vx, vy, vz = particles.vx, particles.vy, particles.vz
    @turbo for i in eachindex(x)
        x[i] += vx[i] * Dt
        y[i] += vy[i] * Dt
        z[i] += vz[i] * Dt
    end
    return particles
end

end


In [ ]:
p10 = I10.Particles(
    Float32[row["x"] for row in raw],
    Float32[row["y"] for row in raw],
    Float32[row["z"] for row in raw],
    Float32[row["vx"] for row in raw],
    Float32[row["vy"] for row in raw],
    Float32[row["vz"] for row in raw],
);


In [ ]:
@btime for _ in 1:timesteps
    I10.advance($p10, $dt)
end




In [ ]:
@btime for _ in 1:timesteps
    I10.advance_turbo($p10, $dt)
end

In [ ]:
state = p10
Profile.clear()
@profile for _ in 1:(1000 * timesteps)
    I10.advance(state, dt)
end
ProfileSVG.view()


In [ ]:
@code_llvm I10.advance(p10, dt)


In [ ]:
@code_llvm I10.advance_turbo(p10, dt)

In [ ]:
# Dane w macierzach
module M1
export Particles, advance

struct Particles
    pos::Matrix{Float32}
    vel::Matrix{Float32}
end

function advance(particles::Particles, dt)
    pos, vel = particles.pos, particles.vel
    Dt = Float32(dt)
    for i in axes(pos, 1)
        p = pos[i, :]
        v = vel[i, :]
        p[1] += v[1] * Dt
        p[2] += v[2] * Dt
        p[3] += v[3] * Dt
        pos[i, :] = p
    end
    return particles
end

end


In [ ]:
m1 = let
    n = length(raw)
    pos = Matrix{Float32}(undef, n, 3)
    vel = Matrix{Float32}(undef, n, 3)
    for i in eachindex(raw)
        row = raw[i]
        pos[i, 1] = Float32(row["x"]); 
        pos[i, 2] = Float32(row["y"]); 
        pos[i, 3] = Float32(row["z"])
        vel[i, 1] = Float32(row["vx"]); 
        vel[i, 2] = Float32(row["vy"]); 
        vel[i, 3] = Float32(row["vz"])
    end
    M1.Particles(pos, vel)
end


In [ ]:
@btime for _ in 1:timesteps
    M1.advance($m1, $dt)
end


In [ ]:
state = m1
Profile.clear()
@profile for _ in 1:(500 * timesteps)
    M1.advance(state, dt)
end
ProfileSVG.view()


In [ ]:
# Widoki
module M2
export Particles, advance

struct Particles
    pos::Matrix{Float32}
    vel::Matrix{Float32}
end

function advance(particles::Particles, dt)
    pos, vel = particles.pos, particles.vel
    Dt = Float32(dt)
    @views for i in axes(pos, 1)
        p = pos[i, :]
        v = vel[i, :]
        p[1] += v[1] * Dt
        p[2] += v[2] * Dt
        p[3] += v[3] * Dt
    end
    return particles
end

end


In [ ]:
m2 = let
    n = length(raw)
    pos = Matrix{Float32}(undef, n, 3)
    vel = Matrix{Float32}(undef, n, 3)
    for i in eachindex(raw)
        row = raw[i]
        pos[i, 1] = Float32(row["x"]); pos[i, 2] = Float32(row["y"]); pos[i, 3] = Float32(row["z"])
        vel[i, 1] = Float32(row["vx"]); vel[i, 2] = Float32(row["vy"]); vel[i, 3] = Float32(row["vz"])
    end
    M2.Particles(pos, vel)
end


In [ ]:
@btime for _ in 1:timesteps
    M2.advance($m2, $dt)
end


In [ ]:
state = m2
Profile.clear()
@profile for _ in 1:(1000 * timesteps)
    M2.advance(state, dt)
end
ProfileSVG.view()

In [ ]:
@code_warntype M2.advance(m2, dt)

In [ ]:
# Funkcje barierowe
module BarrierDemo
export GraphNode, adjoint_relu_naive!, _adjoint_relu_kernel!, adjoint_relu_barrier!

mutable struct GraphNode{OP,N,T}
    args::NTuple{N,GraphNode}
    grad::T
    data::T
end

GraphNode{OP}(args::Tuple, grad::T, data::T) where {OP,T} = GraphNode{OP, length(args), T}(args, grad, data)

function adjoint_relu_naive!(y::GraphNode{:relu,1})
    x, = y.args 
    for i in 1:length(x.data)
        if x.data[i] == y.data[i]
            x.grad[i] += y.grad[i]
        end
    end
    return nothing
end

function _adjoint_relu_kernel!(xgrad, xdata, ydata, ygrad)
    for i in 1:length(xdata)
        if xdata[i] == ydata[i]
            xgrad[i] += ygrad[i]
        end
    end
    return nothing
end

function adjoint_relu_barrier!(y::GraphNode{:relu,1})
    x, = y.args
    _adjoint_relu_kernel!(x.grad, x.data, y.data, y.grad)
    return nothing
end

end


In [ ]:
bn = 4096

bx1 = BarrierDemo.GraphNode{:tensor}((), zeros(bn), randn(bn))
by1 = BarrierDemo.GraphNode{:relu}((bx1,), randn(bn), max.(0.0, bx1.data))

bx2 = BarrierDemo.GraphNode{:tensor}((), zeros(bn), copy(bx1.data))
by2 = BarrierDemo.GraphNode{:relu}((bx2,), copy(by1.grad), copy(by1.data))
nothing

In [ ]:
bench_naive = @benchmark BarrierDemo.adjoint_relu_naive!($by1)

In [ ]:
bench_barrier = @benchmark BarrierDemo.adjoint_relu_barrier!($by2)

In [ ]:
@code_warntype BarrierDemo.adjoint_relu_naive!(by1)

In [ ]:
@code_warntype BarrierDemo.adjoint_relu_barrier!(by1)

In [ ]:
# brak niestabilności w funkcji wykonujcej obliczenia
@code_warntype BarrierDemo._adjoint_relu_kernel!(bx2.grad, bx2.data, by2.data, by2.grad)